## 7. 用 LangChain 实现最小 RAG

### 7.1. 配置模型组件

`HuggingFaceEmbeddings` 使用 `embed_documents` 编码文档，使用 `embed_query` 编码问题。

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

load_dotenv("../.env")
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
if not EMBEDDING_MODEL:
    raise RuntimeError("请在 .env 中配置 EMBEDDING_MODEL。")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")

embeddings = HuggingFaceEmbeddings(
    model=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)
llm = (
    ChatOpenAI(
        model=LLM_MODEL,
        api_key=api_key,
        base_url=base_url or None,
        temperature=0,
    )
    if api_key
    else None
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

### 7.2. 加载并分块文档

先按 Markdown 标题划分章节，再将过长章节切到约 800 字符。前者保留文档结构，后者避免片段包含过多主题。

`chunk_overlap=0` 可能拆开跨边界的语句或表格；增加重叠可以缓解该问题，但会扩大索引并增加重复召回。参数应根据文档结构和检索评测调整。

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

data_dir = Path("../data")
documents = []
for path in sorted(data_dir.rglob("*.md")):
    if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
        continue
    documents.append(
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={"source": path.relative_to(data_dir).as_posix()},
        )
    )

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "标题"), ("##", "章节"), ("###", "小节")],
    strip_headers=False,
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=0)

chunks = []
for document in documents:
    header_chunks = header_splitter.split_text(document.page_content)
    for chunk in header_chunks:
        chunk.metadata.update(document.metadata)
    chunks.extend(text_splitter.split_documents(header_chunks))

print(f"加载文档：{len(documents)} 篇")
print(f"分块结果：{len(chunks)} 段")
print("文档来源：")
for source in sorted({doc.metadata['source'] for doc in documents}):
    print(f"- {source}")

加载文档：14 篇
分块结果：163 段
文档来源：
- 产品/冲锋衣-JK902/产品规格.md
- 产品/冲锋衣-JK902/质检报告.md
- 产品/战术背包-BP701/产品规格.md
- 产品/战术背包-BP701/质检报告.md
- 产品/瑜伽裤-YG301/产品规格.md
- 产品/瑜伽裤-YG301/质检报告.md
- 产品/羊毛开衫-CR502/产品规格.md
- 产品/羊毛开衫-CR502/质检报告.md
- 噪音文档/卖家基础指引.md
- 噪音文档/洗涤常识.md
- 尺码表/女款-2026现行.md
- 尺码表/男款-2026现行.md
- 法规政策/FTC纺织标识法.md
- 法规政策/退货政策-2026现行.md


### 7.3. 构建 Qdrant 检索器

`QdrantVectorStore.from_documents` 完成文档向量化与写库；`location=":memory:"` 表示索引仅在当前进程中保存。`as_retriever(search_kwargs={"k": 10})` 创建返回十个结果的检索器。

In [3]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="fashion_knowledge_langchain",
)
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

In [4]:
question = "SKU-YG301 瑜伽裤的面料成分和防透光要求是什么？"
results = retriever.invoke(question)

for index, doc in enumerate(results, start=1):
    print(f"--- 结果 {index}｜{doc.metadata['source']} ---")
    print(doc.page_content[:500])


--- 结果 1｜产品/瑜伽裤-YG301/产品规格.md ---
# 瑜伽裤 SKU-YG301 技术规格书  
> 文档编号: DOC-PROD-YG301-TECH
> 商品代号: SKU-YG301 (Naked-Feel High-Waist Yoga Leggings)
> 类目: 女士运动瑜伽服
> 性别: 女款
> 季节: 2026 春夏
> 状态: Active  
---
--- 结果 2｜产品/瑜伽裤-YG301/质检报告.md ---
# SGS 质检报告 - 瑜伽裤 SKU-YG301  
> 报告编号: SGS-RPT-YG301-2026
> 商品代号: SKU-YG301
> 检测机构: SGS CSTC Technical Co., Ltd. (Shenzhen Branch)
> 报告日期: 2026-06-20
> 测试样品: M码, 3件 (A/B/C)
> 状态: Active  
---
--- 结果 3｜产品/瑜伽裤-YG301/质检报告.md ---
## 3. 结论与签章  
14项测试全部通过。SKU-YG301防透光达最高5级，纤维成分、安全指标、功能性能均满足要求。  
| 角色 | 姓名 | 签章 |
| :--- | :--- | :--- |
| 检测工程师 | Wang Fang | SGS-CSTC-ST-2026-0620-WF |
| 审核主管 | Zhang Min | SGS-CSTC-ST-2026-0620-ZM |
| 检测机构盖章 | SGS CSTC Technical Co., Ltd. | [SGS Official Seal] |  
报告有效期: 2026-06-20 至 2027-06-19
--- 结果 4｜产品/瑜伽裤-YG301/产品规格.md ---
## 1. 产品概述  
裸感无痕高腰瑜伽裤，定位北美中高端女性运动市场。目标用户为25-40岁瑜伽、普拉提、健身女性，追求裸感穿戴体验与深蹲防透光。  
核心卖点：75% Nylon 66 + 25% Lycra四面弹深蹲零透光；230 GSM双面微磨毛Butter-soft触感；四针六线拼缝无摩擦；加宽高腰收腹隐藏钥匙袋。  
![产品主图](../../images/Apparel/28456.jpg)  
![

### 7.4. 组装 LCEL 问答链

LCEL 使用 `|` 依次连接检索、提示词、模型和字符串解析器。`rag_prompt` 用于检查送入模型的消息，`rag_chain` 返回最终答案。

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[来源：{doc.metadata['source']}]\n{doc.page_content}" for doc in docs
    )

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是服饰箱包知识库助手，只根据资料回答，并指出依据的来源。资料没有答案时回答“现有资料无法回答”，不要猜测。"),
        ("human", "资料：\n{context}\n\n问题：{question}"),
    ]
)

rag_prompt = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
)
print(rag_prompt.invoke(question).to_string()[:2500])

System: 你是服饰箱包知识库助手，只根据资料回答，并指出依据的来源。资料没有答案时回答“现有资料无法回答”，不要猜测。
Human: 资料：
[来源：产品/瑜伽裤-YG301/产品规格.md]
# 瑜伽裤 SKU-YG301 技术规格书  
> 文档编号: DOC-PROD-YG301-TECH
> 商品代号: SKU-YG301 (Naked-Feel High-Waist Yoga Leggings)
> 类目: 女士运动瑜伽服
> 性别: 女款
> 季节: 2026 春夏
> 状态: Active  
---

[来源：产品/瑜伽裤-YG301/质检报告.md]
# SGS 质检报告 - 瑜伽裤 SKU-YG301  
> 报告编号: SGS-RPT-YG301-2026
> 商品代号: SKU-YG301
> 检测机构: SGS CSTC Technical Co., Ltd. (Shenzhen Branch)
> 报告日期: 2026-06-20
> 测试样品: M码, 3件 (A/B/C)
> 状态: Active  
---

[来源：产品/瑜伽裤-YG301/质检报告.md]
## 3. 结论与签章  
14项测试全部通过。SKU-YG301防透光达最高5级，纤维成分、安全指标、功能性能均满足要求。  
| 角色 | 姓名 | 签章 |
| :--- | :--- | :--- |
| 检测工程师 | Wang Fang | SGS-CSTC-ST-2026-0620-WF |
| 审核主管 | Zhang Min | SGS-CSTC-ST-2026-0620-ZM |
| 检测机构盖章 | SGS CSTC Technical Co., Ltd. | [SGS Official Seal] |  
报告有效期: 2026-06-20 至 2027-06-19

[来源：产品/瑜伽裤-YG301/产品规格.md]
## 1. 产品概述  
裸感无痕高腰瑜伽裤，定位北美中高端女性运动市场。目标用户为25-40岁瑜伽、普拉提、健身女性，追求裸感穿戴体验与深蹲防透光。  
核心卖点：75% Nylon 66 + 25% Lycra四面弹深蹲零透光；230 GSM双面微磨毛Butter-soft触感；四针六线拼缝无摩擦；加宽高腰收腹隐藏钥匙袋。  

In [6]:
if llm:
    rag_chain = rag_prompt | llm | StrOutputParser()
    print(rag_chain.invoke(question))
else:
    print("未调用模型：请配置 API 后重新运行")

根据现有资料，SKU-YG301 瑜伽裤的面料成分和防透光要求如下：

**面料成分：**  
- 75% Nylon 66（锦纶/超细聚酰胺）+ 25% Lycra Spandex（莱卡四面弹氨纶）  
- 依据来源：《产品/瑜伽裤-YG301/产品规格.md》中“2.1 纤维成分”及“核心卖点”部分。

**防透光要求：**  
- 防透光等级达到最高 **5级**，在深蹲极限拉伸状态下透光率均小于 2%（具体样品测试值：1.8%、2.0%、1.5%），前弯极限拉伸透光率为 0.8%、1.1%。  
- 依据来源：《产品/瑜伽裤-YG301/质检报告.md》中“2.1 防透光 (SGS Squat-Proof)”及“3. 结论与签章”部分。
